In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('../chest-x-ray-data/indiana_reports_with_projections.csv',index_col=False)

In [4]:
df.head(5)

,uid,MeSH,Problems,image,indication,comparison,findings,impression,filename
0,1,normal,normal,Xray Chest PA and Lateral,Positive TB test,None.,The cardiac silhouette and mediastinum size ar...,Normal chest x-XXXX.,1_IM-0001-4001.dcm.png
1,2,Cardiomegaly/borderline;Pulmonary Artery/enlarged,Cardiomegaly;Pulmonary Artery,"Chest, 2 views, frontal and lateral",Preop bariatric surgery.,None.,Borderline cardiomegaly. Midline sternotomy XX...,No acute pulmonary findings.,2_IM-0652-1001.dcm.png
2,3,normal,normal,Xray Chest PA and Lateral,"rib pain after a XXXX, XXXX XXXX steps this XX...",NaN,NaN,"No displaced rib fractures, pneumothorax, or p...",3_IM-1384-1001.dcm.png
3,4,"Pulmonary Disease, Chronic Obstructive;Bullous...","Pulmonary Disease, Chronic Obstructive;Bullous...","PA and lateral views of the chest XXXX, XXXX a...",XXXX-year-old XXXX with XXXX.,None available,There are diffuse bilateral interstitial and a...,1. Bullous emphysema and interstitial fibrosis...,4_IM-2050-1001.dcm.png
4,5,Osteophyte/thoracic vertebrae/multiple/small;T...,Osteophyte;Thickening;Lung,Xray Chest PA and Lateral,Chest and nasal congestion.,NaN,The cardiomediastinal silhouette and pulmonary...,No acute cardiopulmonary abnormality.,5_IM-2117-1003002.dcm.png


In [5]:
df.iloc[3]['impression']

'1. Bullous emphysema and interstitial fibrosis. 2. Probably scarring in the left apex, although difficult to exclude a cavitary lesion. 3. Opacities in the bilateral upper lobes could represent scarring, however the absence of comparison exam, recommend short interval followup radiograph or CT thorax to document resolution.'

In [6]:
import re    

In [7]:
def remove_x_sequences(text):
    if pd.isna(text):
        return text
    # remove sequences like XXXX, xxx, XXXXXXX etc.
    return re.sub(r'[xX]+', '', text)

In [8]:
df['findings'] = df['findings'].apply(remove_x_sequences)

In [9]:
df['findings']

0       The cardiac silhouette and mediastinum size ar...
1       Borderline cardiomegaly. Midline sternotomy . ...
2                                                     NaN
3       There are diffuse bilateral interstitial and a...
4       The cardiomediastinal silhouette and pulmonary...
                              ...                        
3684    The cardiomediastinal silhouette and pulmonary...
3685    The lungs are clear. Heart size is normal. No ...
3686    Heart size within normal limits. Small, nodula...
3687                                                  NaN
3688                                                  NaN
Name: findings, Length: 3689, dtype: str

In [10]:
df['impression'] = df['impression'].apply(remove_x_sequences)

In [11]:
df['impression']

0                                         Normal chest -.
1                            No acute pulmonary findings.
2       No displaced rib fractures, pneumothora, or pl...
3       1. Bullous emphysema and interstitial fibrosis...
4                   No acute cardiopulmonary abnormality.
                              ...                        
3684    1. Interval resolution of bibasilar airspace d...
3685    Clear lungs. No acute cardiopulmonary abnormal...
3686        No acute findings, no evidence for active TB.
3687        Heart size is normal and the lungs are clear.
3688    The cardiac silhouette is normal in size and c...
Name: impression, Length: 3689, dtype: str

In [12]:
import string

In [13]:
def clean_text_edges(text):
    if not text:
        return text

    if pd.isna(text):
        return text

    text = text.strip()

    # Define punctuation to remove at edges (exclude quotes initially)
    edge_punct = string.punctuation.replace('"', '').replace("'", "")

    # Remove leading punctuation (except quotes)
    text = re.sub(rf'^[{re.escape(edge_punct)}]+', '', text)

    # Remove trailing punctuation (except quotes)
    text = re.sub(rf'[{re.escape(edge_punct)}]+$', '', text)

    # Handle single hyphens at edges specifically
    text = re.sub(r'^-+', '', text)
    text = re.sub(r'-+$', '', text)

    # Clean excessive dots/commas etc. at edges
    text = re.sub(r'^[.,!?;:]+', '', text)
    text = re.sub(r'[.,!?;:]+$', '', text)

    # Preserve quotes only if they wrap the whole string
    if (text.startswith('"') and text.endswith('"')) or \
       (text.startswith("'") and text.endswith("'")):
        return text

    # Otherwise remove stray quotes at edges
    text = re.sub(r"^['\"]+", '', text)
    text = re.sub(r"['\"]+$", '', text)

    return text.strip()

In [14]:
df['findings'] , df['impression'] = df['findings'].apply(clean_text_edges) , df['impression'].apply(clean_text_edges)

In [15]:
# df['findings']
df['impression']

0                                            Normal chest
1                             No acute pulmonary findings
2       No displaced rib fractures, pneumothora, or pl...
3       1. Bullous emphysema and interstitial fibrosis...
4                    No acute cardiopulmonary abnormality
                              ...                        
3684    1. Interval resolution of bibasilar airspace d...
3685    Clear lungs. No acute cardiopulmonary abnormal...
3686         No acute findings, no evidence for active TB
3687         Heart size is normal and the lungs are clear
3688    The cardiac silhouette is normal in size and c...
Name: impression, Length: 3689, dtype: str

In [16]:
json_data = []
for _ , row in df.iterrows():

    impression = row['impression'] if not pd.isna(row['impression']) else "N/A"
    findings = row['findings'] if not pd.isna(row['findings']) else "N/A"
    image = row['filename'] if not pd.isna(row['filename']) else "N/A"

    combined_text = f"Findings: {findings} Impression: {impression}".strip()

    json_data.append({
        'id': row['uid'],
        'text': combined_text,
        'image': image
    })

In [17]:
json_data

[{'id': 1,
  'text': 'Findings: The cardiac silhouette and mediastinum size are within normal limits. There is no pulmonary edema. There is no focal consolidation. There are no  of a pleural effusion. There is no evidence of pneumothora Impression: Normal chest',
  'image': '1_IM-0001-4001.dcm.png'},
 {'id': 2,
  'text': 'Findings: Borderline cardiomegaly. Midline sternotomy . Enlarged pulmonary arteries. Clear lungs. Inferior Impression: No acute pulmonary findings',
  'image': '2_IM-0652-1001.dcm.png'},
 {'id': 3,
  'text': 'Findings: N/A Impression: No displaced rib fractures, pneumothora, or pleural effusion identified. Well-epanded and clear lungs. Mediastinal contour within normal limits. No acute cardiopulmonary abnormality identified',
  'image': '3_IM-1384-1001.dcm.png'},
 {'id': 4,
  'text': 'Findings: There are diffuse bilateral interstitial and alveolar opacities consistent with chronic obstructive lung disease and bullous emphysema. There are irregular opacities in the lef

In [18]:
import json
with open ('../chest-x-ray-data/impression_and_findings.json','w') as f:
    json.dump(obj = json_data, fp = f , indent = 4)    

In [ ]:
pip install 